# Scanpy-compatible interactive UMAP

Scanpy owns preprocessing and UMAP computation. GenomeSpy reads the resulting
`adata.obsm["X_umap"]` coordinates and renders the interactive plot.

This example creates original synthetic counts with illustrative gene names,
computes the embedding explicitly with Scanpy, and then plots it with GenomeSpy.
It requires the `scanpy` extra and does not download data.
If your analysis already computed UMAP, skip straight to the plotting step.
Hover for observation IDs and values; scroll to zoom both panels.

In [ ]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse

# Original synthetic counts: three groups with different marker expression.
rng = np.random.default_rng(0)
group_codes = np.repeat(np.arange(3), 40)
counts = rng.poisson(1, size=(120, 20)).astype(np.float32)
counts[np.arange(120), group_codes] += 8
adata = ad.AnnData(
    sparse.csr_matrix(counts),
    obs=pd.DataFrame(
        {
            "cell_type": pd.Categorical(
                np.array(["A", "B", "C"])[group_codes], categories=["C", "B", "A"]
            )
        },
        index=[f"cell-{i}" for i in range(120)],
    ),
    var=pd.DataFrame(index=["MS4A1"] + [f"gene_{i}" for i in range(1, 20)]),
)

## 1. Compute the embedding with Scanpy

These are explicit analysis calls, separate from plotting. Scanpy normalizes and
log-transforms the counts, computes PCA and a neighbor graph, then optimizes UMAP.
`sc.tl.umap` writes the coordinates to `adata.obsm["X_umap"]`.
Use your own existing analysis pipeline in a real workflow.

In [ ]:
import scanpy as sc

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.pca(adata, n_comps=10, random_state=0)
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=10, random_state=0)
sc.tl.umap(adata, random_state=0)

## 2. Plot the existing embedding with GenomeSpy

`gsc.pl.umap` reads the supplied coordinates; it does not compute or rerun UMAP.
Both panels use the same embedding. The gene panel colors cells by the
log-normalized MS4A1 values prepared above.

GenomeSpy’s `width` and `height` set each plot area in logical pixels.
These native controls keep both panels side by side without Matplotlib
configuration. Axes and legends take additional space.

In [ ]:
import genome_spy.scanpy as gsc

chart = gsc.pl.umap(
    adata,
    color=["cell_type", "MS4A1"],
    size=60,
    ncols=2,
    width=300,
    height=300,
    show=False,
    zoom=True,
)
widget = chart.widget(inline=True, controls=False)
widget

In [ ]:
# The live native widget also exposes the existing Python embed connection.
api = await widget.get_embed_api()
root = await api.views.get("root")
assert root.params is not None
print("Live UMAP connected.")